In [0]:
volume_path = "/Volumes/capstone_project_07/capstone_07_schema/capstone_07_ext_vol"
from pyspark.sql.functions import col, trim, count, when

# 1. Define source paths
cdc_path = f"{volume_path}/cdc_data/document_metadata_changes"
access_path = f"{volume_path}/event_data/document_access_audit.csv"
search_path = f"{volume_path}/event_data/search_query_events.csv"
category_path = f"{volume_path}/lookup_data/azure_sql_seed/document_category_lookup.csv"
role_path = f"{volume_path}/lookup_data/azure_sql_seed/role_permission_lookup.csv"
user_path = f"{volume_path}/master_data/azure_sql_seed/user_master.csv"

# 2. Common CSV reader
def read_csv(path):
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .load(path)
    )

# 3. Load all six datasets
cdc_df = read_csv(cdc_path)
access_df = read_csv(access_path)
search_df = read_csv(search_path)
category_df = read_csv(category_path)
role_df = read_csv(role_path)
user_df = read_csv(user_path)

# 4. Define relationships to validate
relationships = [
    ("CDC owner -> User Master", cdc_df, "owner_user_id", user_df, "user_id"),
    ("CDC category -> Category Lookup", cdc_df, "category_code", category_df, "category_code"),
    ("User role -> Role Permission", user_df, "role_code", role_df, "role_code"),
    ("Access user -> User Master", access_df, "user_id", user_df, "user_id"),
    ("Search user -> User Master", search_df, "user_id", user_df, "user_id"),
    ("Access document -> CDC", access_df, "document_id", cdc_df, "document_id"),
    ("Search document -> CDC", search_df, "document_id", cdc_df, "document_id")
]

# 5. Validate distinct keys and references
results = []

for name, source_df, source_col, target_df, target_col in relationships:

    source_keys = (
        source_df
        .select(trim(col(source_col)).alias("join_key"))
        .filter(col("join_key").isNotNull() & (col("join_key") != ""))
        .distinct()
    )

    target_keys = (
        target_df
        .select(trim(col(target_col)).alias("join_key"))
        .filter(col("join_key").isNotNull() & (col("join_key") != ""))
        .distinct()
    )

    matched = source_keys.join(
        target_keys, "join_key", "inner"
    ).count()

    unmatched = source_keys.join(
        target_keys, "join_key", "left_anti"
    ).count()

    missing = source_df.filter(
        col(source_col).isNull() | (trim(col(source_col)) == "")
    ).count()

    results.append((
        name,
        source_keys.count(),
        matched,
        unmatched,
        missing
    ))

    print(f"\nUnmatched keys: {name}")

    display(
        source_keys.join(
            target_keys, "join_key", "left_anti"
        ).limit(10)
    )

# 6. Display relationship summary
summary_df = spark.createDataFrame(
    results,
    ["relationship", "distinct_source_keys", "matched_keys",
     "unmatched_keys", "missing_source_values"]
)

display(summary_df)

# 7. Create temporary SQL views
for name, df in [
    ("cdc_rel", cdc_df),
    ("access_rel", access_df),
    ("search_rel", search_df),
    ("category_rel", category_df),
    ("role_rel", role_df),
    ("user_rel", user_df)
]:
    df.createOrReplaceTempView(name)

print("Cross-dataset relationship validation completed.")

In [0]:
source_keys.join(target_keys, "join_key", "inner")

In [0]:
source_keys.join(target_keys, "join_key", "left_anti")

In [0]:
display(summary_df)

In [0]:
from pyspark.sql.functions import col, trim, countDistinct

# Helper function to display distinct values from a column
def inspect_keys(title, df, column_name, limit=30):
    print(f"\n{title}")

    df.select(trim(col(column_name)).alias("key")) \
      .filter(col("key").isNotNull() & (col("key") != "")) \
      .distinct() \
      .orderBy("key") \
      .show(limit, truncate=False)


# 1. CDC owner IDs vs User Master IDs
inspect_keys("CDC owner_user_id values", cdc_df, "owner_user_id", 20)
inspect_keys("User Master user_id values", user_df, "user_id", 20)

# 2. CDC category codes vs Category Lookup codes
inspect_keys("CDC category_code values", cdc_df, "category_code", 30)
inspect_keys("Category Lookup category_code values", category_df, "category_code", 30)

# 3. User Master roles vs Role Permission roles
inspect_keys("User Master role_code values", user_df, "role_code", 30)
inspect_keys("Role Permission role_code values", role_df, "role_code", 30)